In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

#### Functions

In [2]:
def get_dtype_and_na(df):
    list_dict_row = []
    for col in tqdm(df.columns):
        str_dtype = df[col].dtype
        flt_propna = df[col].isnull().mean()
        dict_row = {
            'col': col,
            'dtype': str_dtype,
            'propna': flt_propna,
        }
        list_dict_row.append(dict_row)
    df_tmp = pd.DataFrame(list_dict_row)
    df_tmp.sort_values(by='propna', ascending=False, inplace=True)
    # get dtype counts
    df_dtype = df_tmp['dtype'].value_counts().reset_index(name='count')
    return df_tmp, df_dtype

In [ ]:
def plot_dtype_freq(df_dtype, str_output):
    x = df_dtype['dtype'].astype(str)
    y = df_dtype['count']
    int_n = y.sum()
    fig, ax = plt.subplots()
    ax.set_title(f'Column Frequency by Data Type (N = {int_n})')
    ax.bar(x, y)
    ax.set_xlabel('Data Type')
    ax.set_ylabel('Frequency')
    plt.savefig(f'{str_output}/dtype_freq.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_propna(df_tmp, str_output, str_suffix=''):
    x = df_tmp['col']
    y = df_tmp['propna']
    fig, ax = plt.subplots(figsize=(15,5))
    ax.set_title('Proportion Missing by Column')
    ax.set_xlabel('Column')
    ax.set_ylabel('Proportion Missing')
    ax.bar(x, y)
    ax.tick_params(axis='x', rotation=90)
    plt.savefig(f'{str_output}/propna{str_suffix}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def analyze_target(df, str_target, dict_pitch_type, str_output):
    df_tmp = df.copy()
    int_n = df_tmp.shape[0]
    # prop missing
    flt_propna_target = df_tmp[str_target].isnull().mean()
    # map
    df_tmp['target_mapped'] = df_tmp[str_target].map(dict_pitch_type)
    # frequency of pitch type
    df_pitch_type = df_tmp['target_mapped'].value_counts(normalize=True).reset_index(name='count')
    x = df_pitch_type['target_mapped']
    y = df_pitch_type['count']
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title(f'Proportion by Pitch Type (N = {int_n}; Prop NaN = {flt_propna_target:0.4f})')
    ax.set_xlabel('Pitch Type')
    ax.set_ylabel('Proportion')
    ax.bar(x, y)
    ax.tick_params(axis='x', rotation=90)
    plt.savefig(f'{str_output}/target_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_pitch_type_by_count(df_main, str_target, list_str_pitch_type, str_output):
    # create count
    df_main['count'] = df_main['balls'].astype(str) + '-' + df_main['strikes'].astype(str)
    
    # pitch type distribution by count
    df_tmp = pd.crosstab(df_main['count'], df_main[str_target], normalize='index') * 100
    
    # order counts logically
    list_count_order = ['0-0', '0-1', '0-2', '1-0', '1-1', '1-2', '2-0', '2-1', '2-2', '3-0', '3-1', '3-2']
    df_tmp = df_tmp.reindex(list_count_order)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type Distribution by Count')
    ax.set_xlabel('Count (Balls-Strikes)')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_type_by_count.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_pitch_type_by_matchup_handedness(df_main, str_target, list_str_pitch_type, str_output):
    # pitcher hand vs batter hand matchup
    df_main['matchup'] = df_main['p_throws'] + ' vs ' + df_main['stand']
    
    df_tmp = pd.crosstab(df_main['matchup'], df_main[str_target], normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type Distribution by Handedness Matchup (Pitcher vs Batter)')
    ax.set_xlabel('Matchup')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_type_by_handedness.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_pitch_type_by_outs(df_main, str_target, list_str_pitch_type, str_output):
    # pitch type by outs
    df_tmp = pd.crosstab(df_main['outs'], df_main[str_target], normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(8, 5))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type Distribution by Outs')
    ax.set_xlabel('Outs')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_type_by_outs.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_n_pitch_types_by_pitcher(df_main, str_target, str_output):
    # how many distinct pitch types does each pitcher throw?
    df_tmp = (
        df_main.groupby('pitcher_id')[str_target]
        .nunique()
        .reset_index()
        .rename(columns={str_target: 'n_pitch_types'})
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    df_tmp['n_pitch_types'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Number of Distinct Pitch Types per Pitcher')
    ax.set_xlabel('Number of Pitch Types')
    ax.set_ylabel('Number of Pitchers')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_types_per_pitcher.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_top_n_pitchers_mix(df_main, str_target, list_str_pitch_type, str_output):
    int_n_top = 20
    # get top N pitchers with most pitches
    list_top_pitchers = df_main.groupby('pitcher_id').size().nlargest(int_n_top).index.tolist()
    # subset
    df_top = df_main[df_main['pitcher_id'].isin(list_top_pitchers)]
    
    df_tmp = pd.crosstab(df_top['pitcher_id'], df_top[str_target], normalize='index') * 100
    
    # sort by fastball %
    if 'FF' in df_tmp.columns:
        df_tmp = df_tmp.sort_values('FF', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    cols_present = [c for c in list_str_pitch_type if c in df_tmp.columns]
    df_tmp[cols_present].plot(kind='barh', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title(f'Pitch Mix for Top {int_n_top} Pitchers (by volume)')
    ax.set_xlabel('Percentage')
    ax.set_ylabel('Pitcher ID')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/top_pitchers_mix.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_pitch_type_by_inning(df_main, str_target, list_str_pitch_type, str_output):
    # pitch type by inning (cap at 9 for cleanliness)
    df_inning = df_main[df_main['inning'] <= 9].copy()
    df_tmp = pd.crosstab(df_inning['inning'], df_inning[str_target], normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type Distribution by Inning (1-9)')
    ax.set_xlabel('Inning')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_type_by_inning.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_pitch_type_by_runners(df_main, str_target, list_str_pitch_type, str_output):
    # runners on base
    df_main['runners_on'] = (
        df_main['on_1b'].notna().astype(int) +
        df_main['on_2b'].notna().astype(int) +
        df_main['on_3b'].notna().astype(int)
    )
    
    df_tmp = pd.crosstab(df_main['runners_on'], df_main[str_target], normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(8, 5))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type Distribution by Runners on Base')
    ax.set_xlabel('Number of Runners on Base')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_type_by_runners.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_pitch_type_by_pitch_count(df_main, str_target, list_str_pitch_type, str_output):
    # bucket pitcher pitch count into groups
    bins = [0, 25, 50, 75, 100, 125, 200]
    labels = ['1-25', '26-50', '51-75', '76-100', '101-125', '126+']
    df_main['pcount_bucket'] = pd.cut(df_main['pcount_pitcher'], bins=bins, labels=labels, right=True)
    
    df_tmp = pd.crosstab(df_main['pcount_bucket'], df_main[str_target], normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type Distribution by Pitcher Pitch Count (game)')
    ax.set_xlabel('Pitch Count Bucket')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/pitch_type_by_pitch_count.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_first_pitch_type_vs_later(df_main, str_target, list_str_pitch_type, str_output):
    # first pitch vs later pitches
    df_main['is_first_pitch'] = (df_main['pcount_at_bat'] == 1).map({True: 'First Pitch', False: 'Later Pitch'})
    
    df_tmp = pd.crosstab(df_main['is_first_pitch'], df_main[str_target], normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(8, 4))
    df_tmp[list_str_pitch_type].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
    ax.set_title('Pitch Type: First Pitch of At-Bat vs. Later Pitches')
    ax.set_xlabel('')
    ax.set_ylabel('Percentage')
    ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{str_output}/first_pitch_vs_later.png', dpi=150, bbox_inches='tight')
    plt.show()

#### Constants

In [15]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_output = './output'

str_target = 'pitch_type'

# pitch type mapping
dict_pitch_type = {
    'FF': 'Four-seam Fastball',
    'FT': 'Two-seam Fastball',
    'SI': 'Sinker',
    'FC': 'Cutter',
    'SL': 'Slider',
    'CH': 'Changeup',
    'CU': 'Curveball',
    'KC': 'Knuckle Curve',
    'FS': 'Splitter',
    'KN': 'Knuckleball',
    'FA': 'Fastball (generic)',
    'EP': 'Eephus',
    'SC': 'Screwball',
    'FO': 'Forkball',
    'IN': 'Intentional Ball',
    'PO': 'Pitchout',
    'AB': 'At-bat related',
    'UN': 'Unknown',
}

# most common pitch types
list_str_pitch_type = [
    'FF',
    'SL',
    'SI',
    'FT',
    'CH',
    'CU',
    'FC',
    'FS',
]

Bucket: assessment-swish-analytics
Task: 01_eda


#### Output

In [16]:
os.makedirs(str_output, exist_ok=True)

#### Import data

In [17]:
%%time

str_filename = 'pitches'
str_uri = f's3://{str_bucket}/00_data_collection/{str_filename}'
df = pd.read_csv(str_uri)
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)
<timed exec>:3: DtypeWarning: Columns (29,30) have mixed types. Specify dtype option on import or set low_memory=False.


CPU times: user 9.34 s, sys: 1.7 s, total: 11 s
Wall time: 23.3 s


,uid,game_pk,year,date,team_id_b,team_id_p,inning,top,at_bat_num,pcount_at_bat,...,runner7_start,runner7_end,runner7_event,runner7_score,runner7_rbi,runner7_earned,created_at,added_at,modified_at,modified_by
0,14143226,286874,2011,2011-03-31,108,118,1,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
1,14143227,286874,2011,2011-03-31,108,118,1,1,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
2,14143228,286874,2011,2011-03-31,108,118,1,1,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
3,14143229,286874,2011,2011-03-31,108,118,1,1,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
4,14143230,286874,2011,2011-03-31,108,118,1,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
718956,19838192,317073,2011,2011-10-28,140,138,9,1,72,3,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718957,19838193,317073,2011,2011-10-28,140,138,9,1,72,4,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718958,19838194,317073,2011,2011-10-28,140,138,9,1,72,5,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718959,19838195,317073,2011,2011-10-28,140,138,9,1,73,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1


#### Data set description

In [18]:
int_nrows, int_ncols = df.shape
print(f'Rows: {int_nrows}; Columns: {int_ncols}')
flt_propna = df.isna().values.mean()
print(f'Proportion Missing: {flt_propna:0.4f}')

Rows: 718961; Columns: 125
Proportion Missing: 0.4509


#### Other data describers

In [19]:
# date range and games per month
df['date'] = pd.to_datetime(df['date'])
dtm_min = df['date'].min().date()
dtm_max = df['date'].max().date()
int_nunique_games = df['game_pk'].nunique()
int_nunique_pitchers = df['pitcher_id'].nunique()
int_nunique_batters = df['batter_id'].nunique()
print(f'Date range: {dtm_min} to {dtm_max}')
print(f'Unique games: {int_nunique_games}')
print(f'Unique pitchers: {int_nunique_pitchers}')
print(f'Unique batters: {int_nunique_batters}')

Date range: 2011-03-31 to 2011-10-28
Unique games: 2467
Unique pitchers: 662
Unique batters: 936


#### Get data types and proportion missing by column

In [20]:
df_tmp, df_dtype = get_dtype_and_na(df=df)
df_tmp

100%|██████████| 125/125 [00:00<00:00, 196.00it/s]


,col,dtype,propna
119,runner7_rbi,float64,1.0
120,runner7_earned,float64,1.0
105,runner5_rbi,float64,1.0
106,runner5_earned,float64,1.0
72,runner1_id,float64,1.0
...,...,...,...
16,final_balls,int64,0.0
121,created_at,object,0.0
122,added_at,object,0.0
123,modified_at,object,0.0


#### Plot dtype

In [ ]:
plot_dtype_freq(df_dtype=df_dtype, str_output=str_output)

#### Plot missing

In [ ]:
plot_propna(df_tmp=df_tmp, str_output=str_output)

#### Show only the columns with missing

In [ ]:
df_tmp = df_tmp[df_tmp['propna'] > 0].copy()
plot_propna(df_tmp=df_tmp, str_output=str_output, str_suffix='_missing_only')

#### Descriptives

In [24]:
df_tmp = df.describe()
# show
df_tmp

,uid,game_pk,year,date,team_id_b,team_id_p,inning,top,at_bat_num,pcount_at_bat,...,runner6_rbi,runner6_earned,runner7_id,runner7_start,runner7_end,runner7_event,runner7_score,runner7_rbi,runner7_earned,modified_by
count,7.189610e+05,718961.000000,718961.0,718961,718961.000000,718961.000000,718961.000000,718961.000000,718961.000000,718961.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,718961.0
mean,1.701980e+07,288557.611823,2011.0,2011-07-02 09:46:03.222483712,128.787182,128.757296,5.033796,0.508900,39.301513,2.865276,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
min,1.414323e+07,286874.000000,2011.0,2011-03-31 00:00:00,108.000000,108.000000,1.000000,0.000000,1.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
25%,1.557356e+07,287507.000000,2011.0,2011-05-17 00:00:00,115.000000,115.000000,3.000000,0.000000,19.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
50%,1.703106e+07,288129.000000,2011.0,2011-07-02 00:00:00,134.000000,134.000000,5.000000,1.000000,39.000000,3.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
75%,1.845894e+07,288749.000000,2011.0,2011-08-19 00:00:00,141.000000,141.000000,7.000000,1.000000,58.000000,4.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
max,1.983820e+07,317073.000000,2011.0,2011-10-28 00:00:00,158.000000,158.000000,19.000000,1.000000,158.000000,16.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
std,1.665143e+06,3642.210956,0.0,NaN,14.281039,14.306711,2.692326,0.499921,23.320725,1.715068,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


#### Target Analysis (pitch_type)

In [ ]:
analyze_target(
    df=df,
    str_target=str_target,
    dict_pitch_type=dict_pitch_type,
    str_output=str_output,
)

#### Load metadata

In [26]:
# load metadata to identify available features
str_filename = 'pitch_by_pitch_metadata.csv'
str_uri = f's3://{str_bucket}/00_data_collection/{str_filename}'
df_meta = pd.read_csv(
    str_uri,
    encoding='latin1',
)
df_meta

,column_name,available_prior_to_pitch,description
0,uid,Yes,unique id
1,game_pk,Yes,unique game id
2,year,Yes,year
3,date,Yes,date
4,team_id_b,Yes,team_id for the batting team
...,...,...,...
120,runner7_earned,NaN,NaN
121,created_at,NaN,NaN
122,added_at,NaN,NaN
123,modified_at,NaN,NaN


#### Get the features available for modeling (i.e., available_prior_to_pitch == Yes)

In [27]:
list_yes = list(df_meta[df_meta['available_prior_to_pitch'] == 'Yes']['column_name'])
list_no = list(df_meta[df_meta['available_prior_to_pitch'] == 'No']['column_name'])

# print yes
print('Columns we can use:')
for a, col in enumerate(list_yes):
    print(f'{a+1}. {col}')

# print no
print()
print('Columns we cannot use:')
for a, col in enumerate(list_no):
    print(f'{a+1}. {col}')

Columns we can use:
1. uid
2. game_pk
3. year
4. date
5. team_id_b
6. team_id_p
7. inning
8. top
9. at_bat_num
10. pcount_at_bat
11. pcount_pitcher
12. balls
13. strikes
14. fouls
15. outs
16. start_tfs
17. start_tfs_zulu
18. batter_id
19. stand
20. b_height
21. pitcher_id
22. p_throws
23. away_team_runs
24. home_team_runs
25. pitch_id
26. on_1b
27. on_2b
28. on_3b

Columns we cannot use:
1. is_final_pitch
2. final_balls
3. final_strikes
4. final_outs
5. at_bat_des
6. event
7. event2
8. event3
9. event4
10. score
11. pitch_des
12. type
13. pitch_tfs
14. pitch_tfs_zulu
15. x
16. y
17. sv_id
18. start_speed
19. end_speed
20. sz_top
21. sz_bot
22. pfx_x
23. pfx_z
24. px
25. pz
26. x0
27. z0
28. y0
29. vx0
30. vz0
31. vy0
32. ax
33. az
34. ay
35. break_length
36. break_y
37. break_angle
38. pitch_type
39. type_confidence
40. zone
41. nasty
42. spin_dir
43. spin_rate
44. cc


#### Subset to main pitch types for cleaner analysis

In [28]:
df_main = df[df[str_target].isin(list_str_pitch_type)].copy()
df_main

,uid,game_pk,year,date,team_id_b,team_id_p,inning,top,at_bat_num,pcount_at_bat,...,runner7_start,runner7_end,runner7_event,runner7_score,runner7_rbi,runner7_earned,created_at,added_at,modified_at,modified_by
26,14143252,286874,2011,2011-03-31,118,108,1,0,7,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
27,14143253,286874,2011,2011-03-31,118,108,1,0,7,2,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
28,14143254,286874,2011,2011-03-31,118,108,1,0,7,3,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
29,14143255,286874,2011,2011-03-31,118,108,1,0,7,4,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
30,14143256,286874,2011,2011-03-31,118,108,1,0,7,5,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
718956,19838192,317073,2011,2011-10-28,140,138,9,1,72,3,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718957,19838193,317073,2011,2011-10-28,140,138,9,1,72,4,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718958,19838194,317073,2011,2011-10-28,140,138,9,1,72,5,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718959,19838195,317073,2011,2011-10-28,140,138,9,1,73,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1


#### Pitch Type by Count (Balls-Strikes)

In [ ]:
plot_pitch_type_by_count(
    df_main=df_main,
    str_target=str_target,
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Pitch Type by Handedness Matchup

In [ ]:
plot_pitch_type_by_matchup_handedness(
    df_main=df_main,
    str_target=str_target,
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Pitch Type by Outs

In [ ]:
plot_pitch_type_by_outs(
    df_main=df_main,
    str_target=str_target,
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Pitcher Pitch Type Diversity

In [ ]:
plot_n_pitch_types_by_pitcher(
    df_main=df_main,
    str_target=str_target,
    str_output=str_output,
)

#### Top Pitcher Pitch Mix (Top 20 by Volume)

In [ ]:
plot_top_n_pitchers_mix(
    df_main=df_main,
    str_target=str_target, 
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Pitch Type by Inning

In [ ]:
plot_pitch_type_by_inning(
    df_main=df_main, 
    str_target=str_target, 
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Pitch Type by Runners on Base

In [ ]:
plot_pitch_type_by_runners(
    df_main=df_main,
    str_target=str_target,
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Pitch Type by Pitch Count

In [ ]:
plot_pitch_type_by_pitch_count(
    df_main=df_main, 
    str_target=str_target, 
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### First Pitch of At-Bat vs. Later Pitches

In [ ]:
plot_first_pitch_type_vs_later(
    df_main=df_main,
    str_target=str_target,
    list_str_pitch_type=list_str_pitch_type,
    str_output=str_output,
)

#### Baseline Accuracy: Pitcher's Most Common Pitch Type

In [38]:
# baseline 1: always predict the most common pitch type overall
str_most_common = df_main[str_target].mode()[0]
flt_naive_acc = (df_main[str_target] == str_most_common).mean() * 100
print(f'Baseline 1 - Always predict "{str_most_common}": {flt_naive_acc:.1f}% accuracy')

Baseline 1 - Always predict "FF": 34.2% accuracy


In [39]:
# baseline 2: predict each pitcher's most common pitch type
sr_pitcher_mode = df_main.groupby('pitcher_id')[str_target].agg(lambda x: x.mode()[0])
df_main['pitcher_mode'] = df_main['pitcher_id'].map(sr_pitcher_mode)
flt_pitcher_acc = (df_main['pitch_type'] == df_main['pitcher_mode']).mean() * 100
print(f'Baseline 2 - Predict each pitcher\'s most common type: {flt_pitcher_acc:.1f}% accuracy')

Baseline 2 - Predict each pitcher's most common type: 47.4% accuracy


In [40]:
# baseline 3: predict each pitcher's most common pitch type per count
sr_pitcher_count_mode = df_main.groupby(['pitcher_id', 'count'])[str_target].agg(lambda x: x.mode()[0])
df_main['pitcher_count_mode'] = df_main.set_index(['pitcher_id', 'count']).index.map(sr_pitcher_count_mode)
flt_pitcher_count_acc = (df_main[str_target] == df_main['pitcher_count_mode']).mean() * 100
print(f'Baseline 3 - Predict pitcher\'s most common type per count: {flt_pitcher_count_acc:.1f}% accuracy')

Baseline 3 - Predict pitcher's most common type per count: 50.1% accuracy


#### Temporal Coverage (for Train/Test Split Planning)

In [ ]:
# date range and games per month
dtm_min = df_main['date'].min().date()
dtm_max = df_main['date'].max().date()
int_nunique_games = df_main['game_pk'].nunique()
int_nunique_pitchers = df_main['pitcher_id'].nunique()
int_nunique_batters = df_main['batter_id'].nunique()
print(f'Date range: {dtm_min} to {dtm_max}')
print(f'Unique games: {int_nunique_games}')
print(f'Unique pitchers: {int_nunique_pitchers}')
print(f'Unique batters: {int_nunique_batters}')

# pitches per month
df_monthly = df_main.groupby(df_main['date'].dt.to_period('M')).size()
fig, ax = plt.subplots(figsize=(10, 4))
df_monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Pitches per Month')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{str_output}/pitches_per_month.png', dpi=150, bbox_inches='tight')
plt.show()

#### Key EDA Takeaways

1. **Target class imbalance**: FF (fastball) dominates at ~33%, while rare types (KN, EP, SC) are <1%. Consider grouping rare types or dropping them.
2. **Count matters**: Pitchers throw more fastballs on hitter-friendly counts (3-0, 3-1) and more breaking balls on pitcher-friendly counts (0-2, 1-2).
3. **Handedness matchup**: Pitch mix shifts based on L/R pitcher-batter matchup (e.g., more sliders vs same-hand batters).
4. **Pitcher identity is king**: Each pitcher has a unique repertoire. Pitcher-level features (overall mix, mix per count) will be the strongest predictors.
5. **First pitch bias**: Fastballs are overrepresented on the first pitch of an at-bat.
6. **Fatigue**: Pitch mix changes slightly as pitch count climbs (more fastballs early, mix shifts later).
7. **Baselines to beat**: Naive "always FF" ~33%, pitcher-mode ~45-50%, pitcher-mode-per-count ~50%+.